# Echo state networks: observations, autonomous forecasts and baselines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/time-series/echo_state_network.ipynb)

This notebook reproduces every reported table and plot, including reservoir states and the rollout animation. It needs only NumPy, Matplotlib and Pillow on CPU. The synthetic sequence is a discrete delayed recurrence, not a real sensor dataset or a reproduction of the continuous-time Mackey-Glass equation.

## Environment

Tested with Python 3.12, NumPy 2.5.3 and Matplotlib 3.11.2. No XGBoost, scikit-learn, GPU or external dataset is used.

In [ ]:
%pip -q install "numpy>=2,<3" "matplotlib>=3.9,<4" "pillow>=10"

In [ ]:
import os
for variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[variable] = "4"
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

## Synthetic series and chronological boundaries

The training-only mean and standard deviation use observations 0..1799. Feature row t consumes u[t] and targets u[t+1]. Initial fitting uses t=100..1798, validation t=1800..2398, and final refitting t=100..2398. Tests target observations 2401..2999. No target in the test block selects a penalty. A 100-step washout excludes the arbitrary initial state.

In [ ]:
series = np.full(4001, 1.2)
for t in range(18, len(series)-1):
    series[t+1] = series[t] + 0.2*series[t-17]/(1+series[t-17]**10) - 0.1*series[t]
series = series[1000:4000]
training_mean = series[:1800].mean()
training_scale = series[:1800].std()
observations = (series-training_mean)/training_scale
targets = observations[1:]
train_times = np.arange(100,1799)
validation_times = np.arange(1800,2399)
refit_times = np.arange(100,2399)
test_times = np.arange(2400,2999)
assert refit_times.max()+1 < test_times.min()
penalties = [1e-6,1e-4,0.01,1.0]
print("Test targets:", test_times[0]+1, "to", test_times[-1]+1)

## Fixed reservoir and trained readout

The 80-unit reservoir has fixed input/recurrent weights. The leaky tanh update is driven by observed values. Ridge fits only the intercept and the state-to-output weights. Standardisation, seed and the ordering of random draws match the article.

In [ ]:
def collect_reservoir(values, radius, seed, units=80):
    random = np.random.default_rng(seed)
    recurrent = random.normal(size=(units,units)) * (random.random((units,units)) < 0.1)
    recurrent *= radius / np.max(abs(np.linalg.eigvals(recurrent)))
    input_weights = random.uniform(-0.5,0.5,(units,2))
    state = np.zeros(units)
    states = []
    for value in values:
        state = 0.5*state + 0.5*np.tanh(recurrent@state + input_weights@np.array([1,value]))
        states.append(state.copy())
    return np.array(states), recurrent, input_weights

def ridge_readout(design, target, penalty):
    regulariser = np.eye(design.shape[1])
    regulariser[0,0] = 0  # Do not penalise the intercept.
    return np.linalg.solve(design.T@design + penalty*regulariser, design.T@target)

def rmse(prediction, truth):
    return float(np.sqrt(np.mean((np.asarray(prediction)-truth)**2)))

## One-step and autonomous protocols

One-step predictions receive actual observations throughout the test. An autonomous rollout begins with the state warmed by the entire available observed history through u[2400], **not** an empty state or a single-observation warm-up. Predict u[2401] first, then feed that prediction into the reservoir. No later actual value enters this rollout.

In [ ]:
records = []
artifacts = {}
rollouts = {}
for radius in [0.3,0.8,1.2]:
    for seed in [11,22,33]:
        states, recurrent, input_weights = collect_reservoir(observations,radius,seed)
        design = np.column_stack([np.ones(len(observations)),states])
        validation_losses = []
        for penalty in penalties:
            candidate = ridge_readout(design[train_times],targets[train_times],penalty)
            validation_losses.append(np.mean((design[validation_times]@candidate-targets[validation_times])**2))
        selected_penalty = penalties[np.argmin(validation_losses)]
        readout = ridge_readout(design[refit_times],targets[refit_times],selected_penalty)
        one_step = design[test_times]@readout
        state = states[2400].copy()
        forecast = []
        for time in test_times:
            prediction = float(np.r_[1,state]@readout)
            forecast.append(prediction)
            state = 0.5*state + 0.5*np.tanh(recurrent@state + input_weights@np.array([1,prediction]))
        forecast = np.array(forecast)
        truth = targets[test_times]
        records.append({"radius":radius,"seed":seed,"penalty":selected_penalty,
                        "one_step_nrmse":rmse(one_step,truth),"rollout_nrmse":rmse(forecast,truth),
                        "first_100_nrmse":rmse(forecast[:100],truth[:100])})
        if radius == 0.8:
            rollouts[seed] = forecast*training_scale + training_mean
            if seed == 11:
                artifacts = {"states":states,"one_step":one_step*training_scale+training_mean,
                             "truth":truth*training_scale+training_mean, "readout":readout.copy(),
                             "recurrent":recurrent.copy(), "input_weights":input_weights.copy()}
print("Radius  Seed  Penalty  One-step NRMSE  Rollout NRMSE")
for row in records:
    print(f"{row['radius']:.1f}     {row['seed']}    {row['penalty']:.0e}      {row['one_step_nrmse']:.5f}         {row['rollout_nrmse']:.4f}")

## Lagged-ridge and constant-mean references

The linear baseline uses 20 observed lags and the same validation penalty grid. Autonomous lagged ridge starts with its actual 20-observation history through time 2400 and appends only predictions afterwards. The constant reference always predicts the training mean, which is zero in standardised units. It uses no test values to fit its level.

In [ ]:
lag_times = np.arange(19,2999)
lag_design = np.array([np.r_[1,observations[t-19:t+1][::-1]] for t in lag_times])
lag_targets = observations[lag_times+1]
train = lag_times < 1799
valid = (lag_times >= 1800) & (lag_times < 2399)
test = lag_times >= 2400
validation_losses = []
for penalty in penalties:
    candidate = ridge_readout(lag_design[train],lag_targets[train],penalty)
    validation_losses.append(np.mean((lag_design[valid]@candidate-lag_targets[valid])**2))
selected_penalty = penalties[np.argmin(validation_losses)]
lag_readout = ridge_readout(lag_design[lag_times<2399],lag_targets[lag_times<2399],selected_penalty)
lag_one_step = lag_design[test]@lag_readout
history = list(observations[2381:2401])
lag_forecast = []
for time in test_times:
    prediction = float(np.r_[1,np.array(history[-20:])[::-1]]@lag_readout)
    lag_forecast.append(prediction)
    history.append(prediction)
truth = lag_targets[test]
lag_forecast = np.array(lag_forecast)
constant_nrmse = rmse(np.zeros(len(truth)),truth)
print(f"Lagged ridge: one-step {rmse(lag_one_step,truth):.5f}, rollout {rmse(lag_forecast,truth):.4f}")
print(f"Constant training mean: rollout {constant_nrmse:.4f}")
print(f"First100: lagged ridge {rmse(lag_forecast[:100],truth[:100]):.4f}, radius0.8 seed33 {next(r['first_100_nrmse'] for r in records if r['radius']==0.8 and r['seed']==33):.4f}")
lag_forecast_original = lag_forecast*training_scale+training_mean
# Article snippet names refer specifically to radius0.8, seed11 after refitting.
u = observations
states = artifacts["states"]
W = artifacts["recurrent"]
Win = artifacts["input_weights"]
readout = artifacts["readout"]

## Forecasts and horizon dependence

Two radius-0.8 seeds beat lagged ridge over 599 steps; seed11 collapses. The early-horizon panel keeps the observed range readable while the full-horizon panel shows the complete collapse. Legends sit outside the data region.

In [ ]:
fig, axes = plt.subplots(3,1,figsize=(10,9))
steps = np.arange(1,600)
axes[0].plot(steps,artifacts["truth"],color="black",label="Observed")
axes[0].plot(steps,artifacts["one_step"],"--",label="ESN one step, seed11")
for seed in [11,22,33]:
    axes[1].plot(steps[:100],rollouts[seed][:100],label=f"ESN seed{seed}")
axes[1].plot(steps[:100],artifacts["truth"][:100],color="black",label="Observed")
axes[1].plot(steps[:100],lag_forecast_original[:100],"--",label="20-lag ridge")
axes[2].plot(steps,artifacts["truth"],color="black",label="Observed")
for seed in [11,22,33]: axes[2].plot(steps,rollouts[seed],label=f"ESN seed{seed}")
axes[2].plot(steps,lag_forecast_original,"--",label="20-lag ridge")
for ax,title in zip(axes,["Observed-input test","Autonomous first100 steps","Autonomous full599 steps"]):
    ax.set(xlabel="Steps after observation2400",ylabel="Series value",title=title)
    ax.legend(loc="upper left",bbox_to_anchor=(1.01,1),fontsize=9)
plt.tight_layout(); plt.show()

## Radius and state heatmap

Every reservoir has larger autonomous than one-step error. The constant-mean reference makes rollout scale interpretable. Spectral radius below one alone does not certify nonlinear driven-system or closed-loop stability.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for seed in [11,22,33]:
    rows = [r for r in records if r["seed"]==seed]
    axes[0].plot([r["radius"] for r in rows],[r["one_step_nrmse"] for r in rows],"o-",label=f"Seed{seed}")
    axes[1].plot([r["radius"] for r in rows],[r["rollout_nrmse"] for r in rows],"o-",label=f"Seed{seed}")
axes[0].axhline(rmse(lag_one_step,truth),color="black",ls="--",label="20-lag ridge")
axes[1].axhline(rmse(lag_forecast,truth),color="black",ls="--",label="20-lag ridge")
axes[1].axhline(constant_nrmse,color="#999999",ls=":",label="Training mean")
for ax,title in zip(axes,["One-step prediction","599-step rollout"]):
    ax.set(xlabel="Spectral radius",ylabel="NRMSE",title=title); ax.legend(fontsize=9)
axes[1].set_yscale("log"); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(9,4))
image = ax.imshow(artifacts["states"][2400:2600,:30].T,aspect="auto",cmap="RdBu_r",vmin=-1,vmax=1)
ax.set(xlabel="Observed test input step",ylabel="Reservoir unit",title="Fixed reservoir states, radius0.8 seed11")
fig.colorbar(image,ax=ax,label="Activation"); plt.show()

## Autonomous rollout animation

The limits follow values already revealed by the animation, so early agreement remains readable. Thirty-one reveal frames cover 150 steps, followed by a final hold. The full599-step static figure above retains the complete context.

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
observed_line, = ax.plot([],[],color="black",label="Observed")
forecast_line, = ax.plot([],[],color="#227C87",label="ESN seed11")
ax.set(xlim=(0,152),xlabel="Steps after last observation",ylabel="Series value",title="Autonomous forecast:150/150 revealed steps; adaptive limits")
ax.legend(loc="upper left",bbox_to_anchor=(1.01,1))
frames = list(range(0,151,5)) + [150]*8
def reveal_forecast(count):
    if count:
        observed_line.set_data(np.arange(1,count+1),artifacts["truth"][:count])
        forecast_line.set_data(np.arange(1,count+1),rollouts[11][:count])
    seen = np.r_[artifacts["truth"][:max(count,1)],rollouts[11][:max(count,1)]]
    lower,upper = min(0.3,seen.min()),max(1.4,seen.max())
    ax.set_ylim(lower-0.1,upper+0.1)
    ax.set_title(f"Autonomous forecast: {count}/150 revealed steps; adaptive limits")
    return observed_line,forecast_line
animation = FuncAnimation(fig,reveal_forecast,frames=frames,interval=200)
fig.tight_layout()
animation.save("echo-state-rollout.gif",writer=PillowWriter(fps=5))
display(HTML(animation.to_jshtml())); plt.close(fig)

## Exercises and references

1. Choose ridge penalties using autonomous validation rollouts, then evaluate once on the untouched test block.
2. Compare horizon-specific errors at25,100 and599 steps without using test errors to tune the model.
3. Compare different washout lengths while preserving chronological boundaries.

[Jaeger (2001), corrected2010 report](https://www.ai.rug.nl/minds/uploads/EchoStatesTechRep.pdf), especially section5 equation21 for leaky units; [ReservoirPy quick start](https://reservoirpy.readthedocs.io/en/latest/user_guide/quickstart.html).